In [ ]:
import pandas as pd


In [9]:
removel_subscription_numbers_df = pd.read_csv(r"C:\Users\kulareddy\Downloads\removal subscription code check.csv")
print(removel_subscription_numbers_df.shape[0])

1594


In [3]:
header_df = pd.read_excel(r"C:\Users\kulareddy\Downloads\New freq load\SubscriptionHeader_04012026.xlsx")
line_df = pd.read_excel(r"C:\Users\kulareddy\Downloads\New freq load\SubscriptionLine_04012026.xlsx")
asset_df = pd.read_csv(r"C:\Users\kulareddy\Downloads\New freq load\SubscriptionCoveredLevel_04012026.csv")
billlines_df = pd.read_excel(r"C:\Users\kulareddy\Downloads\New freq load\SubscriptionCoveredLevelBillline_04012026.xlsx")

In [ ]:
def normalize_key(series):
    s = series.astype("string")
    s = (
        s.str.replace("\u00A0", "", regex=False)
         .str.replace(r"[\u200B-\u200D\uFEFF]", "", regex=True)
         .str.replace(r"\r|\n|\t", "", regex=True)
         .str.strip()
         .str.replace(r"\.0$", "", regex=True)
    )
    s = s.replace({
        "": pd.NA,
        "nan": pd.NA,
        "None": pd.NA,
        "<NA>": pd.NA
    })
    return s


def require_columns(df, df_name, cols):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"{df_name} is missing columns: {missing}")


# -------------------------------------------------
# 1. Validate required columns
# -------------------------------------------------
require_columns(header_df, "header_df", ["SubscriptionNumber"])
require_columns(line_df, "line_df", ["SubscriptionNumber", "SubscriptionProductPuid"])
require_columns(asset_df, "asset_df", ["SubscriptionProductPuid", "CoveredLevelPuid"])
require_columns(billlines_df, "billlines_df", ["CoveredLevelPuid"])
require_columns(removel_subscription_numbers_df, "removel_subscription_numbers_df", ["SubscriptionNumber"])


# -------------------------------------------------
# 2. Normalize keys
# -------------------------------------------------
remove_subs = set(
    normalize_key(removel_subscription_numbers_df["SubscriptionNumber"]).dropna().tolist()
)

header_sub_clean = normalize_key(header_df["SubscriptionNumber"])
line_sub_clean = normalize_key(line_df["SubscriptionNumber"])
line_puid_clean = normalize_key(line_df["SubscriptionProductPuid"])
asset_puid_clean = normalize_key(asset_df["SubscriptionProductPuid"])
asset_cov_clean = normalize_key(asset_df["CoveredLevelPuid"])
bill_cov_clean = normalize_key(billlines_df["CoveredLevelPuid"])


# -------------------------------------------------
# 3. Remove from header_df by SubscriptionNumber
# -------------------------------------------------
header_remove_mask = header_sub_clean.isin(remove_subs)
header_removed = header_df[header_remove_mask].copy()
header_transformed = header_df[~header_remove_mask].copy()


# -------------------------------------------------
# 4. Remove from line_df by SubscriptionNumber
# -------------------------------------------------
line_remove_mask = line_sub_clean.isin(remove_subs)
line_removed = line_df[line_remove_mask].copy()
line_transformed = line_df[~line_remove_mask].copy()


# -------------------------------------------------
# 5. From removed line_df collect SubscriptionProductPuid
# -------------------------------------------------
removed_subscription_product_puids = set(
    normalize_key(line_removed["SubscriptionProductPuid"]).dropna().tolist()
)


# -------------------------------------------------
# 6. Remove from asset_df by SubscriptionProductPuid
# -------------------------------------------------
asset_remove_mask = asset_puid_clean.isin(removed_subscription_product_puids)
asset_removed = asset_df[asset_remove_mask].copy()
asset_transformed = asset_df[~asset_remove_mask].copy()


# -------------------------------------------------
# 7. From removed asset_df collect CoveredLevelPuid
# -------------------------------------------------
removed_covered_level_puids = set(
    normalize_key(asset_removed["CoveredLevelPuid"]).dropna().tolist()
)


# -------------------------------------------------
# 8. Remove from billlines_df by CoveredLevelPuid
# -------------------------------------------------
billlines_remove_mask = bill_cov_clean.isin(removed_covered_level_puids)
billlines_removed = billlines_df[billlines_remove_mask].copy()
billlines_transformed = billlines_df[~billlines_remove_mask].copy()


# -------------------------------------------------
# 9. Build summary
# -------------------------------------------------
summary_df = pd.DataFrame([
    {
        "File": "header_df",
        "Original Count": len(header_df),
        "Removed Count": len(header_removed),
        "Remaining Count": len(header_transformed)
    },
    {
        "File": "line_df",
        "Original Count": len(line_df),
        "Removed Count": len(line_removed),
        "Remaining Count": len(line_transformed)
    },
    {
        "File": "asset_df",
        "Original Count": len(asset_df),
        "Removed Count": len(asset_removed),
        "Remaining Count": len(asset_transformed)
    },
    {
        "File": "billlines_df",
        "Original Count": len(billlines_df),
        "Removed Count": len(billlines_removed),
        "Remaining Count": len(billlines_transformed)
    },
    {
        "File": "Distinct removed SubscriptionNumber",
        "Original Count": len(remove_subs),
        "Removed Count": len(remove_subs),
        "Remaining Count": pd.NA
    },
    {
        "File": "Distinct removed SubscriptionProductPuid",
        "Original Count": len(removed_subscription_product_puids),
        "Removed Count": len(removed_subscription_product_puids),
        "Remaining Count": pd.NA
    },
    {
        "File": "Distinct removed CoveredLevelPuid",
        "Original Count": len(removed_covered_level_puids),
        "Removed Count": len(removed_covered_level_puids),
        "Remaining Count": pd.NA
    }
])


# -------------------------------------------------
# 10. Save in-progress workbook
# -------------------------------------------------
with pd.ExcelWriter("subscription_removal_inprogress.xlsx", engine="openpyxl") as writer:
    removel_subscription_numbers_df.to_excel(writer, sheet_name="removal_list", index=False)
    header_removed.to_excel(writer, sheet_name="header_removed", index=False)
    line_removed.to_excel(writer, sheet_name="line_removed", index=False)
    asset_removed.to_excel(writer, sheet_name="asset_removed", index=False)
    billlines_removed.to_excel(writer, sheet_name="billlines_removed", index=False)
    summary_df.to_excel(writer, sheet_name="summary", index=False)


# -------------------------------------------------
# 11. Save transformed workbook
# -------------------------------------------------
with pd.ExcelWriter("subscription_removal_transformed.xlsx", engine="openpyxl") as writer:
    header_transformed.to_excel(writer, sheet_name="header_df", index=False)
    line_transformed.to_excel(writer, sheet_name="line_df", index=False)
    asset_transformed.to_excel(writer, sheet_name="asset_df", index=False)
    billlines_transformed.to_excel(writer, sheet_name="billlines_df", index=False)


# -------------------------------------------------
# 12. Print results
# -------------------------------------------------
print("subscription_removal_inprogress.xlsx created")
print("subscription_removal_transformed.xlsx created")
print()
print(summary_df)


subscription_removal_inprogress.xlsx created
subscription_removal_transformed.xlsx created

                                       File  Original Count  Removed Count  \
0                                 header_df            3230           1593   
1                                   line_df            7960           4012   
2                                  asset_df            7960           4012   
3                              billlines_df           20077          13968   
4       Distinct removed SubscriptionNumber            1593           1593   
5  Distinct removed SubscriptionProductPuid            4012           4012   
6         Distinct removed CoveredLevelPuid            4012           4012   

  Remaining Count  
0            1637  
1            3948  
2            3948  
3            6109  
4            <NA>  
5            <NA>  
6            <NA>  


In [ ]:
print("\nShapes:")

print("header_df original:", header_df.shape)
print("header_df final   :", header_transformed.shape)

print("line_df original  :", line_df.shape)
print("line_df final     :", line_transformed.shape)

print("asset_df original :", asset_df.shape)
print("asset_df final    :", asset_transformed.shape)

print("billlines_df original:", billlines_df.shape)
print("billlines_df final   :", billlines_transformed.shape)



Shapes:
header_df original: (3230, 25)
header_df final   : (1637, 25)
line_df original  : (7960, 26)
line_df final     : (3948, 26)
asset_df original : (7960, 5)
asset_df final    : (3948, 5)
billlines_df original: (20077, 16)
billlines_df final   : (6109, 16)
